# v3 — Subduction of the Amazon-plume particles, 1993-2013, all four seeds

The production run is 3D (`AdvectionRK4_3D_substep`), so every trajectory carries a
depth `z` (positive down, metres). Particles are seeded in the top 0-7 m
(median 3.5 m) inside the wet release band off the Amazon mouth and advected for
**185 days**. This notebook asks **how many of them leave the surface layer, when,
where, and how fast**.

It is the `run_2` / v3 counterpart of
`Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/notebooks/07_retroflection_sinkers.ipynb`,
which did the same thing for the *old* run but only for particles that first entered a
retroflection box, and with a 100 m threshold. Three things are different here.

1. **No box.** Subduction is counted anywhere in the domain and the *location* of each
   crossing is mapped, rather than assumed.
2. **The threshold is re-tuned to v3.** The old 100 m cut was calibrated on a run in
   which particles could cut corners below the sea bed. In v3 that pathway is closed
   (the sub-stepping kernel refuses trial steps that land in a zero-velocity cell), the
   deepest particle in the whole archive reaches ~220 m and only ~0.7 % pass 100 m — so
   100 m would count almost nothing. **50 m** is used as the primary threshold, with the
   full threshold ladder (20/30/50/80/100/150/200 m) reported alongside.
3. **Two independent detectors**, because the depth record turns out to contain both a
   slow population-wide deepening and genuine fast events:
   * a **threshold crossing** — `max(z)` after release exceeds `SUBDUCT_THR`, optionally
     required to stay below it for `PERSIST_DAYS` (a *sustained* crossing, not a wiggle);
   * a **rapid-descent event** — the largest depth gain inside any `RAPID_WIN`-day window
     exceeds `RAPID_DZ`. This one is insensitive to the slow drift and isolates real
     downwelling.

Both are computed in the same single streaming pass over the **1534 zarr stores /
15.5 M trajectories**, using the folder's usual pattern (one store per worker process,
nothing concatenated in memory). TEOS-10 `sigma0` at release and at the crossing uses the
recipe of `v2.Buoyancy_sigma0_v3.ipynb` / `particle_diagnostics/0.1.Diagnostics.ipynb`,
evaluated on native observations.

`ndry` / `nrefine` (written by the sub-stepping kernel) are carried through, so any event
that touched a solid boundary can be identified and excluded rather than trusted.

In [ ]:
# --- papermill parameters ---------------------------------------------------
tracks_root  = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2"
seeds        = ["11098723", "45678", "67891", "78876"]   # all four v3 seeds; seeds < 25 are test runs and excluded
out_dir      = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis"
fig_dir      = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/figures"

SUBDUCT_THR   = 50.0          # m; primary "left the surface layer" threshold
THR_LADDER    = [20., 30., 50., 80., 100., 150., 200.]   # reported alongside
PERSIST_DAYS  = 5.0           # a crossing is "sustained" if it stays below SUBDUCT_THR this long
RAPID_WIN     = 10.0          # days; window of the rapid-descent detector
RAPID_DZ      = 50.0          # m; descent within RAPID_WIN that counts as an event

OBS_PER_DAY   = 4             # 6-hourly output
max_obs       = 740           # obs kept per store = 185 d; 137 stores are written with 925 obs,
                              # NaN-padded beyond 739, so this only squares up the age grid
AGE_STEP_OBS  = 20            # age profile sampled every 20 obs = 5 days
n_jobs        = 32            # worker processes streaming the stores
max_stores    = 0             # 0 = all stores; >0 = test mode on the first N stores
n_examples    = 24            # trajectories drawn for the example figures
random_state  = 42
fig_dpi       = 140

events_file   = "v3.subduction_events_v3.parquet"
stores_file   = "v3.subduction_stores_v3.csv"
ageprof_file  = "v3.subduction_ageprofile_v3.npz"


In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import gsw
from joblib import Parallel, delayed
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

warnings.filterwarnings("ignore", category=RuntimeWarning)
os.makedirs(out_dir, exist_ok=True); os.makedirs(fig_dir, exist_ok=True)
os.chdir(out_dir)
print("working in", os.getcwd(), "| gsw", gsw.__version__)

PERSIST_OBS = int(round(PERSIST_DAYS * OBS_PER_DAY))
RAPID_OBS   = int(round(RAPID_WIN   * OBS_PER_DAY))

# house style (same palette as the old 07_retroflection_sinkers notebook)
BLUE, ORANGE, AQUA, PURPLE = "#2a78d6", "#eb6834", "#1baf7a", "#7a4fbf"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "0.88"

def style(ax, xlab="", ylab=""):
    ax.set_xlabel(xlab, fontsize=9, color=MUTED)
    ax.set_ylabel(ylab, fontsize=9, color=MUTED)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("0.8")
    ax.tick_params(colors=MUTED, labelsize=8, length=0)
    ax.set_axisbelow(True)
    return ax

def savefig(fig, name):
    p = Path(fig_dir) / name
    fig.savefig(p, dpi=fig_dpi, bbox_inches="tight")
    print("wrote", p)


## The stores

One release every 5 days, 1993-2013, ~10 100 trajectories each. Seeds below 25
(`tracks_v3_22/23/24`) are short test runs and are not in `seeds`.

In [ ]:
stores = []
for s in seeds:
    ss = sorted((Path(tracks_root) / f"tracks_v3_{s}").glob("Parcels_run_*.zarr"))
    print(f"seed {s:>9}: {len(ss):4d} stores   {ss[0].name[-29:-19]} .. {ss[-1].name[-29:-19]}")
    stores += ss
if max_stores:
    stores = stores[:max_stores]
    print("TEST MODE: using only", len(stores), "stores")
print("total stores:", len(stores))

## The scan

One pass per store. For every particle:

* `zmax` — deepest point of the whole track, and the ladder counts `zmax > t`;
* **crossing**: first observation with `z > SUBDUCT_THR`; its day, position, ambient
  salinity/temperature and `sigma0`; plus the longest *continuous* run below the
  threshold (`run_obs`), which is what makes a crossing "sustained";
* **rapid descent**: `max_k ( z[k+RAPID_OBS] - z[k] )`, its magnitude, day and position.
  Computed for *every* particle, not just the ones that cross, so the two detectors stay
  independent;
* `ndry` / `nrefine` — boundary contact recorded by the sub-stepping kernel;
* an **age profile**: the 50/90/99th depth percentiles of the store's whole population
  every 5 days of particle age. Summed over stores this is the diagnostic that separates
  a slow drift from discrete events.

Only the crossing particles are written out as rows; everything else is returned as
per-store totals, so the output table stays a few million rows rather than 15.5 M.

In [ ]:
def longest_run(below):
    """Longest run of consecutive True per row of a (n, nobs) boolean array."""
    n, m = below.shape
    pad = np.zeros((n, m + 2), bool)
    pad[:, 1:-1] = below
    d = np.diff(pad.astype(np.int8), axis=1)
    out = np.zeros(n, np.int32)
    si, sj = np.where(d == 1)
    _,  ej = np.where(d == -1)
    if len(si):
        np.maximum.at(out, si, ej - sj)        # np.where scans row-major, so the
    return out                                 # i-th start pairs with the i-th end


def sigma0_at(z, lat, lon, sal, temp):
    """TEOS-10 sigma0 on native samples; same recipe as v2.Buoyancy_sigma0_v3."""
    bad = ~(np.isfinite(z) & np.isfinite(sal) & np.isfinite(temp)) | (sal <= 0) | (temp <= 0)
    p  = gsw.p_from_z(-z, lat)                 # z is positive-down depth
    SA = gsw.SA_from_SP(sal, p, lon, lat)
    CT = gsw.CT_from_pt(SA, temp)
    s0 = gsw.sigma0(SA, CT)
    return np.where(bad, np.nan, s0)


def scan_store(path):
    """Per-particle subduction diagnostics for one zarr store."""
    ds = xr.open_zarr(path)
    nobs_disk = ds.sizes["obs"]
    M  = min(max_obs, nobs_disk)          # 137 stores hold 925 obs, NaN-padded past 739
    z  = ds.z.values[:, :M]
    n, nobs = z.shape
    fin  = np.isfinite(z)
    zz   = np.where(fin, z, -np.inf)
    zmax = zz.max(axis=1)
    zmax[~fin.any(axis=1)] = np.nan

    # --- age profile: depth percentiles of this store's population vs age -----
    ks   = np.arange(0, M, AGE_STEP_OBS)   # identical grid for every store -> stackable
    prof = np.nanpercentile(np.where(fin, z, np.nan)[:, ks], [50, 90, 99], axis=0)  # (3, n_ages)

    # --- rapid descent, every particle ---------------------------------------
    dz   = zz[:, RAPID_OBS:] - zz[:, :-RAPID_OBS]
    dz   = np.where(np.isfinite(dz), dz, -np.inf)
    rap  = dz.max(axis=1)
    rapk = dz.argmax(axis=1)                   # window START index
    rap[~np.isfinite(rap)] = np.nan

    below = zz > SUBDUCT_THR
    hit   = below.any(axis=1)
    run   = longest_run(below)
    ndry_all = np.nanmax(ds.ndry.values[:, :M], axis=1)
    nref_all = np.nanmax(ds.nrefine.values[:, :M], axis=1)

    stats = dict(
        store=Path(path).name, path=str(path), n_traj=n,
        n_obs=nobs, n_obs_disk=int(nobs_disk),
        n_valid=int(np.isfinite(zmax).sum()),
        n_hit=int(hit.sum()),
        n_sustained=int(((run >= PERSIST_OBS) & hit).sum()),
        n_rapid=int(np.nansum(rap > RAPID_DZ)),
        n_dry=int(np.nansum(ndry_all > 0)),
        n_refine=int(np.nansum(nref_all > 0)),
        zmax_max=float(np.nanmax(zmax)),
        zmax_p99=float(np.nanpercentile(zmax, 99)),
        zmax_med=float(np.nanmedian(zmax)),
        rap_p99=float(np.nanpercentile(rap, 99)),
        **{f"n_gt{int(t)}": int(np.nansum(zmax > t)) for t in THR_LADDER},
    )

    if not hit.any():
        return stats, None, prof

    # --- rows for the crossing particles only --------------------------------
    idx = np.where(hit)[0]
    lat = ds.lat.values[idx, :M]; lon = ds.lon.values[idx, :M]
    sal = ds.sal.values[idx, :M].astype(np.float64)
    tem = ds.temp.values[idx, :M].astype(np.float64)
    tt  = ds.time.values[idx, :M]
    zs  = z[idx]
    rel = (tt - tt[:, :1]) / np.timedelta64(1, "D")

    r     = np.arange(len(idx))
    first = below[idx].argmax(axis=1)
    jmax  = np.where(np.isfinite(zs), zs, -np.inf).argmax(axis=1)
    krap  = rapk[idx]

    def at(k):
        return zs[r, k], lat[r, k], lon[r, k], sal[r, k], tem[r, k]

    zc, lac, loc_, sc, tc = at(first)
    z0, la0, lo0, s0_, t0_ = at(np.zeros(len(idx), int))

    ev = pd.DataFrame(dict(
        local          = idx.astype(np.int32),
        cross_day      = rel[r, first].astype(np.float32),
        cross_lat      = lac.astype(np.float32),
        cross_lon      = loc_.astype(np.float32),
        max_depth      = zmax[idx].astype(np.float32),
        max_day        = rel[r, jmax].astype(np.float32),
        max_lat        = lat[r, jmax].astype(np.float32),
        max_lon        = lon[r, jmax].astype(np.float32),
        run_obs        = run[idx].astype(np.int32),
        rapid_dz       = rap[idx].astype(np.float32),
        rapid_day      = rel[r, krap].astype(np.float32),
        rapid_lat      = lat[r, krap].astype(np.float32),
        rapid_lon      = lon[r, krap].astype(np.float32),
        release_z      = z0.astype(np.float32),
        sal_release    = np.where(s0_ > 0, s0_, np.nan).astype(np.float32),
        sal_cross      = np.where(sc  > 0, sc,  np.nan).astype(np.float32),
        temp_cross     = np.where(tc  > 0, tc,  np.nan).astype(np.float32),
        sigma0_release = sigma0_at(z0, la0, lo0, s0_, t0_).astype(np.float32),
        sigma0_cross   = sigma0_at(zc, lac, loc_, sc, tc).astype(np.float32),
        ndry           = ndry_all[idx].astype(np.int32),
        nrefine        = nref_all[idx].astype(np.int32),
    ))
    ev["store"] = Path(path).name
    return stats, ev, prof

In [ ]:
t0 = time.time()
res = Parallel(n_jobs=n_jobs, verbose=5)(delayed(scan_store)(p) for p in stores)
print(f"\nstreamed {len(stores)} stores in {time.time()-t0:.0f} s")

st = pd.DataFrame([r[0] for r in res])
ev = pd.concat([r[1] for r in res if r[1] is not None], ignore_index=True)
prof = np.stack([r[2] for r in res])                     # (n_stores, 3, n_ages)

# release date / month / year / season from the store name
rel_date = pd.to_datetime(st.store.str.extract(r"_(\d{4}-\d{2}-\d{2})", expand=False))
st["release"] = rel_date
st["year"]  = rel_date.dt.year
st["month"] = rel_date.dt.month
SEASON = {10:"OND",11:"OND",12:"OND",1:"JFM",2:"JFM",3:"JFM",
          4:"AMJ",5:"AMJ",6:"AMJ",7:"JAS",8:"JAS",9:"JAS"}
st["season"] = st.month.map(SEASON)

ev = ev.merge(st[["store","release","year","month","season"]], on="store", how="left")
ev["sustained"] = ev.run_obs >= PERSIST_OBS
ev["rapid"]     = ev.rapid_dz > RAPID_DZ
ev["dsigma0"]   = ev.sigma0_cross - ev.sigma0_release

ev.to_parquet(events_file, index=False)
st.to_csv(stores_file, index=False)
ages = np.arange(0, prof.shape[-1] * AGE_STEP_OBS, AGE_STEP_OBS) / OBS_PER_DAY
np.savez(ageprof_file, prof=prof, ages=ages, stores=st.store.to_numpy(),
         month=st.month.to_numpy(), year=st.year.to_numpy())
print(f"events {ev.shape} -> {events_file}")
print(f"stores {st.shape} -> {stores_file}")
print(f"age profile {prof.shape} -> {ageprof_file}")

## How often does it happen?

The headline numbers. `n_traj` is every particle released; the ladder is the
fraction whose deepest point passes each threshold.

In [ ]:
N = int(st.n_traj.sum())
V = int(st.n_valid.sum())
print(f"released trajectories : {N:,}   ({len(st)} stores, {st.release.min():%Y-%m-%d} .. {st.release.max():%Y-%m-%d})")
print(f"with a usable depth   : {V:,}   ({100*V/N:.3f}%)")
print(f"run length            : {st.n_obs.max()} obs = {st.n_obs.max()/OBS_PER_DAY:.2f} days\n")

lad = pd.DataFrame({
    "threshold_m": THR_LADDER,
    "particles":   [int(st[f"n_gt{int(t)}"].sum()) for t in THR_LADDER]})
lad["percent"] = 100 * lad.particles / V
print("deepest point of the track exceeds:")
print(lad.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

nh, ns, nr = int(st.n_hit.sum()), int(st.n_sustained.sum()), int(st.n_rapid.sum())
print(f"\nprimary threshold {SUBDUCT_THR:.0f} m")
print(f"  crossed at all              : {nh:>10,}   {100*nh/V:7.3f}%")
print(f"  ... and stayed >{PERSIST_DAYS:.0f} d below   : {ns:>10,}   {100*ns/V:7.3f}%  "
      f"({100*ns/nh:.1f}% of the crossings)")
print(f"rapid descent >{RAPID_DZ:.0f} m in {RAPID_WIN:.0f} d : {nr:>10,}   {100*nr/V:7.3f}%")
print(f"\nboundary contact (should be ~0 in v3)")
print(f"  particles with ndry>0       : {int(st.n_dry.sum()):>10,}   {100*st.n_dry.sum()/V:7.5f}%")
print(f"  particles with nrefine>0    : {int(st.n_refine.sum()):>10,}   {100*st.n_refine.sum()/V:7.3f}%")
print(f"  crossings with ndry>0       : {int((ev.ndry>0).sum()):>10,}   "
      f"{100*(ev.ndry>0).mean():7.5f}% of crossings")
print(f"\ndeepest particle in the archive: {st.zmax_max.max():.1f} m")

### Timing and depth of the crossings

In [ ]:
print(ev[["cross_day","max_day","max_depth","run_obs","rapid_dz","rapid_day"]]
      .describe().round(2).to_string())

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
axes[0].hist(ev.cross_day, bins=40, color=BLUE)
style(axes[0], f"day of first crossing of {SUBDUCT_THR:.0f} m", "particles")
axes[1].hist(ev.max_depth, bins=np.arange(50, 230, 5), color=ORANGE)
axes[1].set_yscale("log"); style(axes[1], "deepest point (m)", "")
axes[2].hist(ev.run_obs / OBS_PER_DAY, bins=40, color=AQUA)
axes[2].set_yscale("log")
axes[2].axvline(PERSIST_DAYS, color=INK, lw=1, ls="--")
style(axes[2], f"longest spell below {SUBDUCT_THR:.0f} m (days)", "")
axes[3].hist(ev.rapid_dz, bins=40, color=PURPLE)
axes[3].set_yscale("log")
axes[3].axvline(RAPID_DZ, color=INK, lw=1, ls="--")
style(axes[3], f"largest descent in {RAPID_WIN:.0f} d (m)", "")
for a in axes:
    a.yaxis.grid(True, color=GRID, lw=.6)
fig.suptitle("Subduction of the plume particles: when, how deep, how long, how fast",
             fontsize=11, color=INK, x=.01, ha="left")
fig.tight_layout(); savefig(fig, "v3.subduction_histograms.png"); plt.show()

## Drift or events? The age profile

This is the control the threshold count needs. `prof` holds the 50/90/99th depth
percentiles of **every** particle (not just the subducting ones) against particle age,
for each store. If the whole population sinks together the crossing count is mostly
measuring a slow downward drift; if the deep tail separates from the median, the
crossings are genuine excursions of a minority.

Splitting by release month is the test for seasonality: a January release reaches day 180
in July, a July release reaches day 180 in January. If the deepening followed winter
mixed-layer deepening the curves would be offset from one another; if it follows particle
**age** they collapse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# (a) population percentiles, all releases
P = np.nanmean(prof, axis=0)                     # (3, n_ages) mean over stores
for k, (lab, col) in enumerate([("median", BLUE), ("90th pct", AQUA), ("99th pct", ORANGE)]):
    axes[0].plot(ages, P[k], color=col, lw=1.8, label=lab)
axes[0].axhline(SUBDUCT_THR, color=INK, lw=1, ls="--")
axes[0].annotate(f"{SUBDUCT_THR:.0f} m", (ages[-1], SUBDUCT_THR), xytext=(-30, 4),
                 textcoords="offset points", fontsize=8, color=INK)
axes[0].invert_yaxis(); axes[0].legend(frameon=False, fontsize=8)
style(axes[0], "particle age (days)", "depth (m)")
axes[0].set_title("Depth of the whole population against age", fontsize=10,
                  color=INK, loc="left", pad=8)

# (b) 99th percentile by release month
cmap = plt.get_cmap("twilight")
for m in range(1, 13):
    sel = st.month.to_numpy() == m
    if sel.sum():
        axes[1].plot(ages, np.nanmean(prof[sel, 2], axis=0),
                     color=cmap((m - 1) / 12), lw=1.4, label=f"{m:02d}")
axes[1].axhline(SUBDUCT_THR, color=INK, lw=1, ls="--")
axes[1].invert_yaxis()
axes[1].legend(frameon=False, fontsize=7, ncol=4, title="release month",
               title_fontsize=7)
style(axes[1], "particle age (days)", "99th-percentile depth (m)")
axes[1].set_title("Same curve, split by release month", fontsize=10,
                  color=INK, loc="left", pad=8)
for a in axes:
    a.yaxis.grid(True, color=GRID, lw=.6)
fig.tight_layout(); savefig(fig, "v3.subduction_age_profile.png"); plt.show()

spread = np.array([np.nanmean(prof[st.month.to_numpy() == m, 2], axis=0)[-1]
                   if (st.month.to_numpy() == m).any() else np.nan
                   for m in range(1, 13)])
spread = spread[np.isfinite(spread)]
print(f"99th-pct depth at day {ages[-1]:.0f} by release month: "
      f"{spread.min():.1f} .. {spread.max():.1f} m  (spread {np.ptp(spread):.1f} m)")
print("month of deepest / shallowest:", int(np.nanargmax(spread)) + 1, "/", int(np.nanargmin(spread)) + 1)

## Where does it happen?

Crossing positions binned on the same ~1 deg lon/lat grid the rest of this folder uses
(`v2.bins_v3.npz`), so these maps overlay the count and monthly-mean maps directly.
Left: where particles first pass the threshold. Right: where the rapid-descent events
occur. The release band off the Amazon mouth is marked.

In [ ]:
b = np.load("v2.bins_v3.npz")
lon_bins, lat_bins = b["lon_bins"], b["lat_bins"]
lon_c = .5 * (lon_bins[1:] + lon_bins[:-1]); lat_c = .5 * (lat_bins[1:] + lat_bins[:-1])
print(f"grid {len(lon_c)} x {len(lat_c)} cells, "
      f"{lon_bins[0]:.1f}..{lon_bins[-1]:.1f} E, {lat_bins[0]:.1f}..{lat_bins[-1]:.1f} N")

REL_LON, REL_LAT = (-49.92, -48.33), (-0.17, 1.33)     # release band of the v3 runs
EXT = [-92, -20, -5, 32]

def map_ax(ax, title):
    ax.set_extent(EXT, crs=ccrs.PlateCarree())
    ax.add_feature(cf.LAND, facecolor="0.92", zorder=2)
    ax.coastlines(resolution="50m", lw=.4, color="0.55", zorder=3)
    ax.add_patch(plt.Rectangle((REL_LON[0], REL_LAT[0]),
                               REL_LON[1]-REL_LON[0], REL_LAT[1]-REL_LAT[0],
                               fill=False, ec=AQUA, lw=1.6, zorder=6,
                               transform=ccrs.PlateCarree()))
    gl = ax.gridlines(draw_labels=True, lw=.3, color="0.85", alpha=.7)
    gl.top_labels = gl.right_labels = False
    gl.xformatter, gl.yformatter = LongitudeFormatter(), LatitudeFormatter()
    gl.xlabel_style = gl.ylabel_style = {"size": 7, "color": MUTED}
    ax.set_title(title, fontsize=10, color=INK, loc="left", pad=8)

fig, axes = plt.subplots(1, 2, figsize=(15, 5),
                         subplot_kw=dict(projection=ccrs.PlateCarree()))
for ax, (lo, la, ttl) in zip(axes, [
        (ev.cross_lon, ev.cross_lat,
         f"first crossing of {SUBDUCT_THR:.0f} m  (n={len(ev):,})"),
        (ev.loc[ev.rapid, "rapid_lon"], ev.loc[ev.rapid, "rapid_lat"],
         f"rapid descent >{RAPID_DZ:.0f} m / {RAPID_WIN:.0f} d  (n={int(ev.rapid.sum()):,})")]):
    H, _, _ = np.histogram2d(lo, la, bins=[lon_bins, lat_bins])
    map_ax(ax, ttl)
    pm = ax.pcolormesh(lon_bins, lat_bins, np.ma.masked_less(H, 1).T,
                       norm=LogNorm(), cmap="Spectral_r", zorder=1,
                       transform=ccrs.PlateCarree())
    cb = fig.colorbar(pm, ax=ax, orientation="horizontal", pad=.07, fraction=.045)
    cb.set_label("particles per cell", fontsize=8, color=MUTED)
    cb.ax.tick_params(colors=MUTED, labelsize=7, length=0)
fig.tight_layout(); savefig(fig, "v3.subduction_where.png"); plt.show()

### The same maps, as a share of everyone who passed through

A raw count of crossings is partly just a map of where the particles are. Dividing by the
number of particle *tracks* that visit each cell gives the conditional probability:
**given that a particle reached this cell, how likely was it to be subducting here.**
The denominator is built from the crossing particles' own tracks plus a matched random
sample of non-crossing tracks, so it costs one extra light pass.

In [ ]:
SAMPLE_STORES = min(200, len(stores))          # stores used for the denominator pass
sub = [stores[i] for i in np.linspace(0, len(stores) - 1, SAMPLE_STORES).astype(int)]
NCELL = len(lon_c) * len(lat_c)

def _cellflat(lon, lat):
    """Flat cell index of each (lon, lat), -1 outside the grid."""
    ii = np.digitize(lon, lon_bins) - 1
    jj = np.digitize(lat, lat_bins) - 1
    ok = (ii >= 0) & (ii < len(lon_c)) & (jj >= 0) & (jj < len(lat_c))
    return np.where(ok, ii * len(lat_c) + jj, -1)

def visit_hist(path):
    """Denominator: cells VISITED, counted once per particle (a particle that
    loiters in a cell must not count twice).  Numerator: the cell in which a
    particle makes its FIRST crossing of SUBDUCT_THR.  The ratio is then the
    probability that a particle passing through a cell subducts *in that cell*."""
    ds = xr.open_zarr(path)
    M  = min(max_obs, ds.sizes["obs"])
    z  = ds.z.values[:, :M]
    lo = ds.lon.values[:, :M]; la = ds.lat.values[:, :M]
    n  = z.shape[0]

    flat = _cellflat(lo, la)
    part = np.repeat(np.arange(n), flat.shape[1])
    f    = flat.ravel()
    keep = f >= 0
    # one count per (particle, cell) pair
    pairs = np.unique(part[keep].astype(np.int64) * NCELL + f[keep])
    Hall  = np.bincount(pairs % NCELL, minlength=NCELL)

    zz      = np.where(np.isfinite(z), z, -np.inf)
    below   = zz > SUBDUCT_THR
    crossed = below.any(axis=1)
    Hcross  = np.zeros(NCELL, np.int64)
    if crossed.any():
        k  = np.where(crossed)[0]
        fc = flat[k, below[k].argmax(axis=1)]
        fc = fc[fc >= 0]
        Hcross = np.bincount(fc, minlength=NCELL)
    shape = (len(lon_c), len(lat_c))
    return Hall.reshape(shape), Hcross.reshape(shape)

t0 = time.time()
hv = Parallel(n_jobs=n_jobs, verbose=5)(delayed(visit_hist)(p) for p in sub)
Hall   = np.sum([h[0] for h in hv], axis=0)
Hcross = np.sum([h[1] for h in hv], axis=0)
print(f"denominator pass over {len(sub)} stores in {time.time()-t0:.0f} s")
print(f"visiting particle-cell pairs {Hall.sum():,} | first crossings located {Hcross.sum():,}")

MIN_VISITS = 500
frac = np.where(Hall >= MIN_VISITS, 100 * Hcross / np.maximum(Hall, 1), np.nan)

fig, ax = plt.subplots(figsize=(9.5, 5.2), subplot_kw=dict(projection=ccrs.PlateCarree()))
map_ax(ax, f"probability that a particle passing through a cell first crosses "
           f"{SUBDUCT_THR:.0f} m THERE\n({len(sub)} stores, cells visited by >={MIN_VISITS} particles)")
pm = ax.pcolormesh(lon_bins, lat_bins, np.ma.masked_invalid(frac).T,
                   cmap="magma_r", zorder=1, transform=ccrs.PlateCarree())
cb = fig.colorbar(pm, ax=ax, orientation="horizontal", pad=.07, fraction=.045)
cb.set_label("% of visiting particles that subduct in this cell", fontsize=8, color=MUTED)
cb.ax.tick_params(colors=MUTED, labelsize=7, length=0)
fig.tight_layout(); savefig(fig, "v3.subduction_where_conditional.png"); plt.show()

### Named regions — is the retroflection where it happens?

The obvious place to look is the NBC retroflection, which is the box the old
`07_retroflection_sinkers` notebook conditioned on. Two numbers per region: the **share**
of all crossings that occur there, and the **traffic-normalised rate** — crossings in the
region divided by the particles that pass through it — which removes the "there are simply
more particles here" effect.

In [ ]:
REGIONS = {
    "plume / Amazon shelf":      dict(lon=(-52, -44), lat=(-2,  6)),
    "NBC retroflection (old box)": dict(lon=(-48, -44), lat=( 5,  8)),
    "NBC / Guiana corridor":     dict(lon=(-58, -48), lat=( 6, 12)),
    "eastern Caribbean":         dict(lon=(-68, -60), lat=(10, 16)),
    "central/W Caribbean":       dict(lon=(-78, -68), lat=(12, 20)),
    "N of Hispaniola / open Atl": dict(lon=(-76, -66), lat=(18, 24)),
}

def in_box(lo, la, bx):
    return ((lo > bx["lon"][0]) & (lo < bx["lon"][1])
            & (la > bx["lat"][0]) & (la < bx["lat"][1]))

rows = []
for name, bx in REGIONS.items():
    m  = in_box(ev.cross_lon, ev.cross_lat, bx)
    mr = in_box(ev.rapid_lon, ev.rapid_lat, bx) & ev.rapid
    ci = (lon_c > bx["lon"][0]) & (lon_c < bx["lon"][1])
    cj = (lat_c > bx["lat"][0]) & (lat_c < bx["lat"][1])
    vis = Hall[np.ix_(ci, cj)].sum()
    crs = Hcross[np.ix_(ci, cj)].sum()
    rows.append(dict(region=name,
                     crossings=int(m.sum()),
                     pct_of_all_crossings=100 * m.mean(),
                     rapid=int(mr.sum()),
                     pct_of_all_rapid=100 * mr.sum() / max(int(ev.rapid.sum()), 1),
                     visiting_particles=int(vis),
                     rate_per_visitor_pct=100 * crs / max(vis, 1),
                     median_day=float(ev.loc[m, "cross_day"].median()) if m.any() else np.nan,
                     median_depth=float(ev.loc[m, "max_depth"].median()) if m.any() else np.nan))
reg = pd.DataFrame(rows)
print(reg.to_string(index=False, float_format=lambda x: f"{x:9.3f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
y = np.arange(len(reg))
axes[0].barh(y, reg.pct_of_all_crossings, color=BLUE)
axes[0].set_yticks(y); axes[0].set_yticklabels(reg.region, fontsize=8)
axes[0].invert_yaxis(); style(axes[0], "% of all 50 m crossings", "")
axes[1].barh(y, reg.rate_per_visitor_pct, color=ORANGE)
axes[1].set_yticks(y); axes[1].set_yticklabels([])
axes[1].invert_yaxis(); style(axes[1], "% of the particles visiting the region that subduct there", "")
for a in axes:
    a.xaxis.grid(True, color=GRID, lw=.6)
fig.suptitle("Where the subduction happens, raw and normalised by traffic",
             fontsize=11, color=INK, x=.01, ha="left")
fig.tight_layout(); savefig(fig, "v3.subduction_regions.png"); plt.show()

reg.to_csv("v3.subduction_regions_v3.csv", index=False)
print("\nsaved v3.subduction_regions_v3.csv")

### The statistic the region table gets wrong

`pct_of_all_crossings` above is dominated by **traffic**: a region only a few per cent of
the particles ever visit cannot contribute many crossings, however strongly those few
subduct. And a *first* crossing is charged once, to one place, so a region where particles
dip below the threshold repeatedly still scores once per particle.

The question "does subduction happen **here**" is better answered by conditioning on the
particles that enter a region and asking how deep they get **while they are inside it**.
That is what this cell does, over the same `sub` sample, on a ladder of thresholds — 50 m
is tuned to the deep Caribbean behaviour and is too blunt for the shallower tropics, so
20 and 30 m matter here.

The median age of the particles while they are in each region is reported alongside,
because it is the confound: the retroflection is crossed young and shallow, the Caribbean
is reached old and deep.

In [ ]:
DEPTH_LADDER = [10, 20, 30, 50, 80]

def region_depths(path):
    """Deepest point reached by each particle WHILE INSIDE each region."""
    ds = xr.open_zarr(path)
    M  = min(max_obs, ds.sizes["obs"])
    z  = ds.z.values[:, :M]
    lo = ds.lon.values[:, :M]; la = ds.lat.values[:, :M]
    zz = np.where(np.isfinite(z), z, -np.inf)
    age = np.arange(M) / OBS_PER_DAY
    out = {}
    for name, bx in REGIONS.items():
        m = in_box(lo, la, bx)
        inside = m.any(axis=1)
        if not inside.any():
            out[name] = (0, np.zeros(len(DEPTH_LADDER)), np.nan, np.nan)
            continue
        zmax_in = np.where(m, zz, -np.inf).max(axis=1)[inside]
        age_in  = np.nanmean(np.where(m, age[None, :], np.nan)[inside], axis=1)
        out[name] = (int(inside.sum()),
                     np.array([(zmax_in > t).sum() for t in DEPTH_LADDER]),
                     float(np.nanmedian(age_in)), float(np.nanmedian(zmax_in)))
    return out

t0 = time.time()
rd = Parallel(n_jobs=n_jobs, verbose=5)(delayed(region_depths)(p) for p in sub)
print(f"region-depth pass over {len(sub)} stores in {time.time()-t0:.0f} s")

rows = []
for name in REGIONS:
    n = sum(r[name][0] for r in rd)
    c = np.sum([r[name][1] for r in rd], axis=0)
    a = [r[name][2] for r in rd if np.isfinite(r[name][2])]
    d = [r[name][3] for r in rd if np.isfinite(r[name][3])]
    row = dict(region=name, entering=n,
               median_age_in_region_d=np.median(a) if a else np.nan,
               median_maxdepth_inside_m=np.median(d) if d else np.nan)
    row.update({f"pct_gt{t}m": 100 * cc / max(n, 1) for t, cc in zip(DEPTH_LADDER, c)})
    rows.append(row)
regd = pd.DataFrame(rows)
print("\n% of the particles ENTERING each region whose deepest point WHILE INSIDE exceeds:")
print(regd.to_string(index=False, float_format=lambda x: f"{x:9.2f}"))
regd.to_csv("v3.subduction_region_depths_v3.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
w = 0.15
for i, t in enumerate(DEPTH_LADDER):
    ax.bar(np.arange(len(regd)) + (i - 2) * w, regd[f"pct_gt{t}m"], width=w,
           label=f">{t} m", color=plt.get_cmap("viridis")(i / (len(DEPTH_LADDER) - 1)))
ax.set_xticks(range(len(regd)))
ax.set_xticklabels(regd.region, fontsize=7, rotation=18, ha="right")
ax.set_yscale("log"); ax.legend(frameon=False, fontsize=8, ncol=5)
ax.yaxis.grid(True, color=GRID, lw=.6)
style(ax, "", "% of entering particles reaching this depth inside the region")
ax.set_title("Subduction conditioned on presence: the retroflection is weak, "
             "but it is not zero", fontsize=10, color=INK, loc="left", pad=8)
fig.tight_layout(); savefig(fig, "v3.subduction_region_depths.png"); plt.show()

## Seasonality and interannual variation

By **release** month and year — the same grouping the rest of the folder uses. Rates are
per store so that the uneven number of releases per year cannot bias them.

In [ ]:
rate = (st.groupby("month")[["n_hit","n_sustained","n_rapid","n_valid"]].sum())
for c in ("n_hit","n_sustained","n_rapid"):
    rate[c + "_pct"] = 100 * rate[c] / rate.n_valid

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].bar(rate.index, rate.n_hit_pct, color=BLUE, width=.7)
style(axes[0], "release month", f"% crossing {SUBDUCT_THR:.0f} m")
axes[0].set_xticks(range(1, 13))
axes[1].bar(rate.index, rate.n_rapid_pct, color=PURPLE, width=.7)
style(axes[1], "release month", f"% with a >{RAPID_DZ:.0f} m / {RAPID_WIN:.0f} d descent")
axes[1].set_xticks(range(1, 13))

yr = st.groupby("year")[["n_hit","n_rapid","n_valid"]].sum()
axes[2].plot(yr.index, 100*yr.n_hit/yr.n_valid, "o-", color=BLUE, ms=3, lw=1.3,
             label=f"crossed {SUBDUCT_THR:.0f} m")
axes[2].plot(yr.index, 100*yr.n_rapid/yr.n_valid, "o-", color=PURPLE, ms=3, lw=1.3,
             label="rapid descent")
axes[2].legend(frameon=False, fontsize=8)
style(axes[2], "release year", "% of particles")
for a in axes:
    a.yaxis.grid(True, color=GRID, lw=.6)
fig.suptitle("Subduction rate by release month and year", fontsize=11, color=INK, x=.01, ha="left")
fig.tight_layout(); savefig(fig, "v3.subduction_seasonality.png"); plt.show()

print(rate[["n_hit_pct","n_sustained_pct","n_rapid_pct"]].round(2).to_string())

## What kind of water subducts?

`sigma0` at release against `sigma0` at the crossing. The particles start in the plume
(fresh, light) and the ambient water they sit in at the crossing is far denser, so the
descent is **not** along a conserved isopycnal — the T/S carried by the particle is
sampled from the Eulerian field, and the plume signal is mixed away long before the
particle leaves the surface layer.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].hist(ev.sigma0_release.dropna(), bins=60, color=AQUA, alpha=.85, label="at release")
axes[0].hist(ev.sigma0_cross.dropna(),   bins=60, color=ORANGE, alpha=.85, label="at crossing")
axes[0].legend(frameon=False, fontsize=8)
style(axes[0], "sigma0 (kg m-3)", "particles")

s = ev.sample(min(60000, len(ev)), random_state=random_state)
axes[1].scatter(s.sal_release, s.sal_cross, s=2, alpha=.15, color=BLUE, linewidths=0)
axes[1].plot([0, 38], [0, 38], color=INK, lw=.8, ls="--")
axes[1].set_xlim(0, 38); axes[1].set_ylim(30, 38)
style(axes[1], "salinity at release", "salinity at crossing")

axes[2].scatter(s.cross_day, s.max_depth, s=2, alpha=.15, color=PURPLE, linewidths=0)
style(axes[2], "day of first crossing", "deepest point (m)")
for a in axes:
    a.yaxis.grid(True, color=GRID, lw=.6)
fig.suptitle("Density and salinity of the subducting particles", fontsize=11,
             color=INK, x=.01, ha="left")
fig.tight_layout(); savefig(fig, "v3.subduction_water_mass.png"); plt.show()

print(f"sigma0 at release : median {ev.sigma0_release.median():.2f} kg m-3")
print(f"sigma0 at crossing: median {ev.sigma0_cross.median():.2f} kg m-3")
print(f"change            : median {ev.dsigma0.median():+.2f} kg m-3")
print(f"salinity at release / crossing: {ev.sal_release.median():.2f} / {ev.sal_cross.median():.2f}")

## Pick some and look at them

`n_examples` trajectories drawn from the **strongest** events — the largest rapid
descent, so the descent is legible — spread across seeds and release months rather than
taken from one store. Their full tracks are re-read from the zarr stores.

In [ ]:
cand = ev[ev.rapid & (ev.ndry == 0)].copy()
if len(cand) == 0:
    cand = ev[ev.ndry == 0].copy()
# spread the picks over release months and seeds, then take the biggest descent in each group
cand["seed"] = cand.store.str.extract(r"Parcels_run_(\d+)_", expand=False)
pick = (cand.sort_values("rapid_dz", ascending=False)
            .groupby(["seed", "month"], group_keys=False).head(1)
            .sort_values("rapid_dz", ascending=False)
            .head(n_examples))
print(f"{len(pick)} examples, descent {pick.rapid_dz.min():.0f}..{pick.rapid_dz.max():.0f} m "
      f"in {RAPID_WIN:.0f} d, deepest point {pick.max_depth.min():.0f}..{pick.max_depth.max():.0f} m")
print(pick[["store","local","cross_day","rapid_dz","rapid_day","max_depth","max_lat","max_lon"]]
      .to_string(index=False))

store_by_name = {p.name: p for p in stores}

def full_track(store, locals_):
    ds = xr.open_zarr(store_by_name[store])
    loc = np.asarray(locals_)
    M = min(max_obs, ds.sizes["obs"])
    z = ds.z.values[loc, :M]; la = ds.lat.values[loc, :M]; lo = ds.lon.values[loc, :M]
    sa = ds.sal.values[loc, :M].astype(np.float64); te = ds.temp.values[loc, :M].astype(np.float64)
    tt = ds.time.values[loc, :M]
    rel = (tt - tt[:, :1]) / np.timedelta64(1, "D")
    out = []
    for k in range(len(loc)):
        ok = np.isfinite(lo[k]) & np.isfinite(z[k]) & ~np.isnat(tt[k])
        if ok.sum() < 2:
            continue
        out.append(dict(store=store, local=int(loc[k]), day=rel[k][ok],
                        lat=la[k][ok], lon=lo[k][ok], z=z[k][ok],
                        sigma0=sigma0_at(z[k][ok], la[k][ok], lo[k][ok],
                                         sa[k][ok], te[k][ok])))
    return out

tracks = [t for s, g in pick.groupby("store") for t in full_track(s, g.local.to_numpy())]
print(f"\nloaded {len(tracks)} full tracks")

In [ ]:
fig = plt.figure(figsize=(15, 8.5))
gs = fig.add_gridspec(2, 2, height_ratios=[1.45, 1], hspace=.25, wspace=.18)
axm = fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree())
axz = fig.add_subplot(gs[1, 0])
axs = fig.add_subplot(gs[1, 1])

map_ax(axm, f"{len(tracks)} of the strongest subduction events, path coloured by depth")
axm.set_aspect("auto")   # fill the gridspec row instead of leaving whitespace beside it
vmax = float(np.nanpercentile(np.concatenate([t["z"] for t in tracks]), 99))
for t in tracks:
    axm.plot(t["lon"], t["lat"], "-", color="0.7", lw=.5, zorder=3,
             transform=ccrs.PlateCarree())
    sc = axm.scatter(t["lon"], t["lat"], c=t["z"], cmap="viridis_r", s=3,
                     vmin=0, vmax=vmax, zorder=4, linewidths=0,
                     transform=ccrs.PlateCarree())
# mark each track's deepest point
for t in tracks:
    k = int(np.nanargmax(t["z"]))
    axm.plot(t["lon"][k], t["lat"][k], "o", mfc="none", mec=ORANGE, mew=1.2, ms=7,
             zorder=6, transform=ccrs.PlateCarree())
cb = fig.colorbar(sc, ax=axm, pad=.02, fraction=.03)
cb.set_label("depth (m)", fontsize=8, color=MUTED)
cb.ax.tick_params(colors=MUTED, labelsize=7, length=0)

for t in tracks:
    axz.plot(t["day"], t["z"], "-", color=BLUE, lw=.9, alpha=.7)
    axs.plot(t["sigma0"], t["z"], "-", color=PURPLE, lw=.9, alpha=.7)
axz.axhline(SUBDUCT_THR, color=ORANGE, lw=1.2, ls="--")
axz.invert_yaxis(); style(axz, "days since release", "depth (m)")
axz.set_title("depth against time", fontsize=10, color=INK, loc="left", pad=6)
axs.axhline(SUBDUCT_THR, color=ORANGE, lw=1.2, ls="--")
axs.invert_yaxis(); style(axs, "sigma0 (kg m-3)", "depth (m)")
axs.set_title("depth against potential density", fontsize=10, color=INK, loc="left", pad=6)
for a in (axz, axs):
    a.yaxis.grid(True, color=GRID, lw=.6)
savefig(fig, "v3.subduction_examples.png"); plt.show()

### Four of them on their own

The same tracks, but one panel each for the four strongest, so the descent is readable
against its own map rather than in the pile above.

In [ ]:
top4 = sorted(tracks, key=lambda t: -np.nanmax(t["z"]))[:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 7),
                         gridspec_kw=dict(height_ratios=[1.4, 1], hspace=.3, wspace=.25))
for c, t in enumerate(top4):
    ax = fig.add_subplot(2, 4, c + 1, projection=ccrs.PlateCarree())
    axes[0, c].remove()
    pad = 3
    ax.set_extent([np.nanmin(t["lon"])-pad, np.nanmax(t["lon"])+pad,
                   np.nanmin(t["lat"])-pad, np.nanmax(t["lat"])+pad],
                  crs=ccrs.PlateCarree())
    ax.add_feature(cf.LAND, facecolor="0.92", zorder=2)
    ax.coastlines(resolution="50m", lw=.4, color="0.55", zorder=3)
    gl = ax.gridlines(draw_labels=True, lw=.3, color="0.85")
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"size": 6, "color": MUTED}
    s = ax.scatter(t["lon"], t["lat"], c=t["z"], cmap="viridis_r", s=5, linewidths=0,
                   zorder=4, transform=ccrs.PlateCarree())
    k = int(np.nanargmax(t["z"]))
    ax.plot(t["lon"][k], t["lat"][k], "o", mfc="none", mec=ORANGE, mew=1.4, ms=9,
            zorder=6, transform=ccrs.PlateCarree())
    ax.plot(t["lon"][0], t["lat"][0], "*", color=AQUA, ms=10, zorder=6,
            transform=ccrs.PlateCarree())
    ax.set_title(f"{t['store'][12:21]} #{t['local']}\nmax {np.nanmax(t['z']):.0f} m "
                 f"on day {t['day'][k]:.0f}", fontsize=8, color=INK, loc="left")
    fig.colorbar(s, ax=ax, fraction=.04, pad=.02).ax.tick_params(labelsize=6, length=0)

    a = axes[1, c]
    a.plot(t["day"], t["z"], "-", color=BLUE, lw=1.2)
    a.axhline(SUBDUCT_THR, color=ORANGE, lw=1, ls="--")
    a.invert_yaxis(); a.yaxis.grid(True, color=GRID, lw=.6)
    style(a, "days since release", "depth (m)" if c == 0 else "")
fig.suptitle("The four deepest examples: star = release, circle = deepest point",
             fontsize=11, color=INK, x=.01, ha="left")
savefig(fig, "v3.subduction_examples_individual.png"); plt.show()

## What this run does and does not show

Read the numbers above with these four things in mind.

1. **The 185-day run length truncates the answer.** The crossing-day histogram is skewed
   hard towards the end of the run and the age profile is still deepening at day 185, so
   every rate quoted here is a *lower bound* — a longer integration would count more.
2. **Age, not season, sets the deepening.** The release-month curves collapse onto one
   another, so what the threshold count mostly measures is how far the population has
   spread downward by a given particle age, not a winter mixed-layer signal.
3. **That is why the rapid-descent detector is here.** It is the part of the signal that
   the slow drift cannot produce, and it should be preferred when quoting "how often
   subduction happens".
4. **Boundary contact is not the explanation in v3.** `ndry` is essentially zero across
   the archive, so unlike the old run these deep excursions are not particles cutting
   corners below the sea bed. `nrefine > 0` is common and harmless — it only means the
   sub-stepping kernel refined its step somewhere along the track.

Outputs written by this notebook:

| file | contents |
|---|---|
| `v3.subduction_events_v3.parquet` | one row per particle that crosses `SUBDUCT_THR`: crossing day/position, deepest point, spell length, rapid-descent magnitude/day/position, sigma0 and salinity at release and crossing, `ndry`/`nrefine`, release date/month/year/season |
| `v3.subduction_stores_v3.csv` | per-store totals: trajectories, threshold-ladder counts, crossings, sustained, rapid, boundary contacts, depth percentiles |
| `v3.subduction_ageprofile_v3.npz` | `prof` (n_stores, 3, n_ages) 50/90/99th depth percentiles vs age, with `ages`, `month`, `year` |
| `figures/v3.subduction_*.png` | the figures above |